# CAPs Clustering Evaluation Metrics

Calculate evaluation metrics for 100 permutations of k-means clustering (k values range = 2-15).

In [ ]:
import os
from dotenv import load_dotenv
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics.cluster import adjusted_rand_score
import warnings
from tqdm import tqdm
from time import time

load_dotenv()

base_dir = os.environ['BASE_DIR']
runs_str = "run-ON"
output_dir = os.path.join(base_dir, f"coactivation_patterns_{runs_str}")

## 1 - Silhouette score, Calinski-Harabasz Index, Davies-Bouldin Index, Inertia (SSE)
**Metrics computed:**
- **Silhouette Score** — measures how similar each point is to its own cluster vs. neighboring clusters. Ranges from -1 to 1; higher is better.
- **Calinski-Harabasz Index** — ratio of between-cluster to within-cluster variance. Higher values indicate better-defined clusters.
- **Davies-Bouldin Index** — average similarity between each cluster and its most similar cluster. Lower values indicate better separation.
- **Inertia (SSE)** — sum of squared distances of samples to their closest cluster center. Used for elbow method; lower is better.<br>

Analysis Function

In [ ]:
def preform_kmeans_and_save_metrics(CAP_TS, n_permutations=100, k_range=range(2, 16), output_dir=None):
    """
    Apply K-means clustering and calculate multiple clustering evaluation metrics for different k values.
    Saves the results for all permutations in numpy arrays in a dedicated metrics directory.

    Parameters:
    -----------
    CAP_TS : numpy array (n_timepoints x n_ROIs)
        The timeseries data
    n_permutations : int
        Number of permutations (for random seeds)
    k_range : range
        The range of k values to evaluate
    output_dir : str
        Directory to save results
    """
    start_time = time()

    # Create metrics directory if it doesn't exist
    metrics_dir = os.path.join(output_dir, 'eval_metrics')
    if not os.path.exists(metrics_dir):
        os.makedirs(metrics_dir)
        print(f"Created metrics directory: {metrics_dir}")

    # Create subdirectories for intermediate results
    intermediate_dir = os.path.join(metrics_dir, 'intermediate_results')
    if not os.path.exists(intermediate_dir):
        os.makedirs(intermediate_dir)
        print(f"Created intermediate results directory: {intermediate_dir}")

    # Initialize storage for results
    silhouette_scores_array = np.zeros((n_permutations, len(k_range)))
    calinski_scores_array = np.zeros((n_permutations, len(k_range)))
    davies_scores_array = np.zeros((n_permutations, len(k_range)))
    inertia_array = np.zeros((n_permutations, len(k_range)))

    for perm in tqdm(range(n_permutations), desc="Running permutations"):
        random_state = perm

        for idx, k in enumerate(k_range):
            kmeans = KMeans(
                n_clusters=k,
                init='k-means++',
                max_iter=1000,
                n_init='auto',
                random_state=random_state
            )

            kmeans.fit(CAP_TS)
            cluster_labels = kmeans.labels_

            # Calculate all metrics
            sil_score = silhouette_score(CAP_TS, cluster_labels)
            calinski_score = calinski_harabasz_score(CAP_TS, cluster_labels)
            davies_score = davies_bouldin_score(CAP_TS, cluster_labels)
            inertia = kmeans.inertia_

            # Store results
            silhouette_scores_array[perm, idx] = sil_score
            calinski_scores_array[perm, idx] = calinski_score
            davies_scores_array[perm, idx] = davies_score
            inertia_array[perm, idx] = inertia

        # Save intermediate results every 10 permutations
        if (perm + 1) % 10 == 0:
            np.save(os.path.join(intermediate_dir, f'silhouette_scores_perm_{perm+1}.npy'), silhouette_scores_array[:perm+1])
            np.save(os.path.join(intermediate_dir, f'calinski_scores_perm_{perm+1}.npy'), calinski_scores_array[:perm+1])
            np.save(os.path.join(intermediate_dir, f'davies_scores_perm_{perm+1}.npy'), davies_scores_array[:perm+1])
            np.save(os.path.join(intermediate_dir, f'inertia_perm_{perm+1}.npy'), inertia_array[:perm+1])

    # Save final results with descriptive filenames
    np.save(os.path.join(metrics_dir, 'silhouette_scores_all_permutations.npy'), silhouette_scores_array)
    np.save(os.path.join(metrics_dir, 'calinski_harabasz_scores_all_permutations.npy'), calinski_scores_array)
    np.save(os.path.join(metrics_dir, 'davies_bouldin_scores_all_permutations.npy'), davies_scores_array)
    np.save(os.path.join(metrics_dir, 'inertia_scores_all_permutations.npy'), inertia_array)

    # Save summary statistics
    summary_stats = {
        'silhouette': {
            'mean': np.mean(silhouette_scores_array, axis=0),
            'std': np.std(silhouette_scores_array, axis=0)
        },
        'calinski': {
            'mean': np.mean(calinski_scores_array, axis=0),
            'std': np.std(calinski_scores_array, axis=0)
        },
        'davies': {
            'mean': np.mean(davies_scores_array, axis=0),
            'std': np.std(davies_scores_array, axis=0)
        },
        'inertia': {
            'mean': np.mean(inertia_array, axis=0),
            'std': np.std(inertia_array, axis=0)
        }
    }

    for metric, stats in summary_stats.items():
        np.save(os.path.join(metrics_dir, f'{metric}_mean.npy'), stats['mean'])
        np.save(os.path.join(metrics_dir, f'{metric}_std.npy'), stats['std'])

    end_time = time()
    duration = end_time - start_time
    print(f"Process completed in: {duration:.2f} seconds")
    print(f"Results saved in: {metrics_dir}")

    return silhouette_scores_array, calinski_scores_array, davies_scores_array, inertia_array

Run the analysis function with 100 permutations

In [ ]:
# Load data
CAP_TS = np.load(os.path.join(output_dir, f"CAP_TS_{runs_str}_zscored.npy"))

# Run the analysis
k_range = range(2, 16)
silhouette_scores, calinski_scores, davies_scores, inertia = preform_kmeans_and_save_metrics(
    CAP_TS,
    n_permutations=100,
    k_range=k_range,
    output_dir=output_dir
)

 ## 2- Adjusted Rand Index mertic calculation

In [ ]:
# Filter the warning about CPU cores
import warnings
warnings.filterwarnings("ignore", message="Could not find the number of physical cores")

def calculate_ari_stability(data, k_values, n_iterations=100, subsample_ratio=0.8, random_state=42):
    """
    Calculate clustering stability using Adjusted Rand Index across multiple subsamples.

    Parameters:
    -----------
    data : numpy array (n_timepoints x n_ROIs)
        The timeseries data
    k_values : list or range
        List of k values to evaluate
    n_iterations : int
        Number of iterations for each k value
    subsample_ratio : float
        Proportion of data to use in each subsample (0.0-1.0)
    random_state : int or None
        Random seed for reproducibility

    Returns:
    --------
    stability_scores : dict
        Dictionary with k values as keys and mean ARI stability scores as values
    all_scores : dict
        Dictionary with k values as keys and lists of all pairwise ARI scores for each k
    """
    n_samples = data.shape[0]
    subsample_size = int(n_samples * subsample_ratio)

    stability_scores = {}
    all_scores = {}

    for k in k_values:
        print(f"Calculating ARI stability for k={k}")

        # Generate n_iterations different clusterings
        all_labels = []
        all_indices = []

        for i in tqdm(range(n_iterations), desc=f"Iterations for k={k}"):
            # Randomly sample data points
            iter_rng = np.random.RandomState(i)
            indices = iter_rng.choice(n_samples, size=subsample_size, replace=False)
            indices.sort()  # Sort for consistent comparison
            all_indices.append(indices)

            # Run k-means
            kmeans = KMeans(
                n_clusters=k,
                init='k-means++',
                max_iter=1000,
                n_init='auto',
                random_state=i
            )

            labels = kmeans.fit_predict(data[indices])
            all_labels.append(labels)

        # Calculate pairwise ARI between all pairs of clusterings
        ari_scores = []

        for i in range(n_iterations):
            for j in range(i+1, n_iterations):
                # Find common indices between the two subsamples
                common_indices = np.intersect1d(all_indices[i], all_indices[j])

                # Map back to positions in each subsample
                idx_i = np.searchsorted(all_indices[i], common_indices)
                idx_j = np.searchsorted(all_indices[j], common_indices)

                # Extract labels for common points
                labels_i = all_labels[i][idx_i]
                labels_j = all_labels[j][idx_j]

                # Calculate ARI
                if len(np.unique(labels_i)) > 1 and len(np.unique(labels_j)) > 1:
                    ari = adjusted_rand_score(labels_i, labels_j)
                    ari_scores.append(ari)

        # Store mean ARI score
        stability_scores[k] = np.mean(ari_scores)
        all_scores[k] = ari_scores

    return stability_scores, all_scores

In [ ]:
load_dotenv()
base_dir = os.environ['BASE_DIR']
runs_str = "run-ON"
output_dir = os.path.join(base_dir, f"coactivation_patterns_{runs_str}")

# Create a directory for ARI stability results 
ari_dir = os.path.join(output_dir, 'eval_metrics', 'ari_stability')
os.makedirs(ari_dir, exist_ok=True)

# Run ARI stability analysis
print("\nRunning ARI stability analysis...")
start_time = time()  # Start the timer

k_range = range(2, 16)
stability_scores, all_scores = calculate_ari_stability(
    CAP_TS,
    k_range,
    n_iterations=100,  # Can be adjusted based on computational resources
    subsample_ratio=0.8,
    random_state=42
    )

end_time = time()
duration = end_time - start_time
print(f"ARI stability analysis completed in: {duration:.2f} seconds")

In [ ]:
# Save results
if ari_dir:
    np.save(os.path.join(ari_dir, 'ari_stability_scores.npy'),
            np.array([(k, stability_scores[k]) for k in k_range],
                     dtype=[('k', int), ('ari', float)]))
    
# Save all ARI scores for each k
for k in k_range:
    np.save(os.path.join(ari_dir, f'ari_all_scores_k{k}.npy'),
            np.array(all_scores[k]))